# Benchmarking CNN and ViT Models under Partial Image Masking

This work compares the robustness of a Convolutional Neural Network (CNN) and a Vision Transformer (ViT) when evaluated on the FER-2013 dataset with varying levels of image masking.

## Methodology
- **Models:**  
  - **CNN:** ResNet-based classifier.  
  - **ViT:** Pretrained Vision Transformer fine-tuned for FER.  
- **Masking:**  
  - Images are resized to `224×224`, block-masked at different ratios, then resized to `48×48`.  
  - Mask ratios tested: `0.0` to `0.8` in steps of `0.1`.  
  - Mask block sizes tested: `10, 20, 30, 40, 50` pixels.  
- **Evaluation:**  
  - Accuracy and cross-entropy loss computed for each mask configuration.  
  - For **mask ratio = 0**, results are computed once and reused to avoid redundant computation.  

## Output
For each mask block size:
- **Accuracy vs Mask Ratio** plot with CNN and ViT curves.
- **Loss vs Mask Ratio** plot with CNN and ViT curves.

This allows us to visualize and compare the resilience of CNN and ViT architectures to partial occlusion of facial expressions.


# Importing the needed packages

In [1]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset,random_split, WeightedRandomSampler
import matplotlib.pyplot as plt
import torch.optim as optim
import numpy as np
from timm.models.vision_transformer import VisionTransformer
import timm
import os
from tqdm import tqdm
from PIL import Image
from torch.utils.data import ConcatDataset
import torch
from torchvision import datasets
import timm
import copy
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import random
import torch.nn.functional as F

# Models Definitions

## CNN model

In [2]:
class FERResNet18(nn.Module):
    def __init__(self, num_classes):
        super(FERResNet18, self).__init__()
        self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

## ViT model
### MAE model

In [3]:
def create_mae_model_from_timm(new_depth=6):
    # Step 1: Create ViT encoder with same architecture, without downloading
    encoder = timm.create_model(
        'vit_base_patch16_224',
        pretrained=False,  # Prevents downloading
        img_size=224,
        patch_size=16,
        embed_dim=768,
        depth=new_depth,
        num_heads=12,
        mlp_ratio=4.0
    )

    # Step 2: Load local checkpoint (make sure it's compatible with your encoder config)
    ckpt_path = '../models/vit_base_patch16_224.pth'
    state_dict = torch.load(ckpt_path, map_location='cpu')
    
    # If needed, remove "module." prefix (some checkpoints have it)
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

    # Load weights
    missing_keys, unexpected_keys = encoder.load_state_dict(state_dict, strict=False)
    print("✅ Loaded encoder weights.")
    print(f"Missing keys: {len(missing_keys)} | Unexpected keys: {len(unexpected_keys)}")

    return encoder


class SimpleMAE(torch.nn.Module):
    def __init__(
        self, 
        encoder,
        decoder_embed_dim=512,
        decoder_depth=8,
        decoder_num_heads=16,
        mask_ratio=0.75,
        norm_pix_loss=True
    ):
        super().__init__()
        self.encoder = encoder
        self.img_size = encoder.patch_embed.img_size[0]
        self.patch_size = encoder.patch_embed.patch_size[0]
        self.embed_dim = encoder.embed_dim
        self.num_patches = (self.img_size // self.patch_size) ** 2
        self.mask_token = torch.nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_embed = torch.nn.Linear(self.embed_dim, decoder_embed_dim, bias=True)
        self.decoder_pos_embed = torch.nn.Parameter(torch.zeros(1, self.num_patches + 1, decoder_embed_dim))
        self.decoder_blocks = torch.nn.ModuleList([
            timm.models.vision_transformer.Block(
                decoder_embed_dim, decoder_num_heads, 4.0
            ) for _ in range(decoder_depth)
        ])
        
        self.decoder_norm = torch.nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = torch.nn.Linear(
            decoder_embed_dim, self.patch_size**2 * 3, bias=True
        )
        
        self.mask_ratio = mask_ratio
        self.norm_pix_loss = norm_pix_loss
        self.initialize_weights()
    
    def initialize_weights(self):
        torch.nn.init.normal_(self.mask_token, std=0.02)
        torch.nn.init.normal_(self.decoder_pos_embed, std=0.02)
        torch.nn.init.xavier_uniform_(self.decoder_embed.weight)
        torch.nn.init.zeros_(self.decoder_embed.bias)
        torch.nn.init.xavier_uniform_(self.decoder_pred.weight)
        torch.nn.init.zeros_(self.decoder_pred.bias)
    
    def random_masking(self, x, mask_ratio):
        N, L, D = x.shape
        len_keep = int(L * (1 - mask_ratio))
        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(
            x, dim=1, 
            index=ids_keep.unsqueeze(-1).repeat(1, 1, D)
        )
        mask = torch.ones([N, L], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        
        return x_masked, mask, ids_restore
    
    def forward_encoder(self, x, mask_ratio):
        patches = self.encoder.patch_embed(x)
        patches = patches + self.encoder.pos_embed[:, 1:, :]
        patches_masked, mask, ids_restore = self.random_masking(patches, mask_ratio)
        cls_token = self.encoder.cls_token + self.encoder.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(patches_masked.shape[0], -1, -1)
        x = torch.cat((cls_tokens, patches_masked), dim=1)
        for blk in self.encoder.blocks:
            x = blk(x)
        x = self.encoder.norm(x)
        
        return x, mask, ids_restore
    
    def forward_decoder(self, x, ids_restore):
        x = self.decoder_embed(x)
        mask_tokens = self.mask_token.repeat(
            x.shape[0], ids_restore.shape[1] + 1 - x.shape[1], 1
        )
        x_ = x[:, 1:, :]
        x_ = torch.cat([x_, mask_tokens], dim=1)
        x_ = torch.gather(
            x_, dim=1,
            index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2])
        )
        x = torch.cat([x[:, :1, :], x_], dim=1)
        x = x + self.decoder_pos_embed
        for blk in self.decoder_blocks:
            x = blk(x)
        x = self.decoder_norm(x)
        x = self.decoder_pred(x)
        x = x[:, 1:, :]
        
        return x
    
    def patchify(self, imgs):
        p = self.patch_size
        h = w = self.img_size // p
        x = imgs.reshape(shape=(imgs.shape[0], 3, h, p, w, p))
        x = torch.einsum('nchpwq->nhwpqc', x)
        x = x.reshape(shape=(imgs.shape[0], h * w, p**2 * 3))
        
        return x
    
    def unpatchify(self, x):
        p = self.patch_size
        h = w = int(x.shape[1]**0.5)
        assert h * w == x.shape[1]
        x = x.reshape(shape=(x.shape[0], h, w, p, p, 3))
        x = torch.einsum('nhwpqc->nchpwq', x)
        imgs = x.reshape(shape=(x.shape[0], 3, h * p, w * p))
        
        return imgs
    
    def forward_loss(self, imgs, pred, mask):
        target = self.patchify(imgs)
        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1.e-6)**0.5
        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()
        
        return loss
    
    def forward(self, imgs, mask_ratio=None):
        if mask_ratio is None:
            mask_ratio = self.mask_ratio
        latent, mask, ids_restore = self.forward_encoder(imgs, mask_ratio)
        pred = self.forward_decoder(latent, ids_restore)
        loss = self.forward_loss(imgs, pred, mask)
        return loss, pred, mask, ids_restore

### ViT classifier model

In [4]:
class FERClassifier(nn.Module):
    def __init__(self, mae_encoder, num_classes=7):
        super().__init__()
        self.encoder = mae_encoder
        self.classifier = nn.Sequential(
            nn.LayerNorm(mae_encoder.embed_dim),
            nn.Linear(mae_encoder.embed_dim, num_classes)
        )

    def forward(self, x):
        # Patchify & embed: (B, 3, 224, 224) -> (B, N_patches+1, embed_dim)
        tokens = self.encoder.patch_embed(x)
        cls_token = self.encoder.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat((cls_token, tokens), dim=1)
        x = x + self.encoder.pos_embed[:, :x.size(1), :]
        x = self.encoder.pos_drop(x)

        for blk in self.encoder.blocks:
            x = blk(x)

        x = self.encoder.norm(x)
        cls_embedding = x[:, 0]  # Take [CLS] token

        return self.classifier(cls_embedding)

# Loading the test data from the FER 2013 dataset

In [5]:
test_dir = "../datasets/FER_2013_enhanced/preprocessed_test"
# Loading the data for the Vit FER classifier
# setting up the ViT transform
vit_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # deterministic resize for val/test
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
# setting up the CNN transform
cnn_test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

vit_test_dataset = datasets.ImageFolder(root=test_dir, transform=vit_test_transform)
cnn_test_dataset = datasets.ImageFolder(root=test_dir, transform=cnn_test_transform)

# Loading the models

In [6]:
cnn_model = FERResNet18(num_classes=7)
encoder = create_mae_model_from_timm(new_depth=6)
vit_model = FERClassifier(encoder, num_classes=7)
cnn_model.load_state_dict(torch.load("../models/CNN_FER.pth"))
vit_model.load_state_dict(torch.load("../models/fer_model_transfered_knowledge.pth")["model_state_dict"])

✅ Loaded encoder weights.
Missing keys: 0 | Unexpected keys: 72


<All keys matched successfully>

# Evaluation functions

In [7]:
def remap_labels(dataset):
    desired_class_order = [
        "neutral",   # 0
        "happy",     # 1
        "sad",       # 2
        "surprise",  # 3
        "fear",      # 4
        "disgust",   # 5
        "anger"      # 6
    ]
    
    # Original FER-2013 class order
    original_class_order = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    
    # Create a mapping from original index to new index
    label_remap = {
        original_class_order.index('neutral'): desired_class_order.index('neutral'),
        original_class_order.index('happy'): desired_class_order.index('happy'),
        original_class_order.index('sad'): desired_class_order.index('sad'),
        original_class_order.index('surprise'): desired_class_order.index('surprise'),
        original_class_order.index('fear'): desired_class_order.index('fear'),
        original_class_order.index('disgust'): desired_class_order.index('disgust'),
        original_class_order.index('angry'): desired_class_order.index('anger')
    }
    # Create new list of samples with remapped labels
    new_samples = []
    for path, label in dataset.samples:
        new_label = label_remap[label]
        new_samples.append((path, new_label))
    
    # Create new dataset with remapped labels
    new_dataset = datasets.ImageFolder(root=dataset.root, transform=dataset.transform)
    new_dataset.samples = new_samples
    new_dataset.targets = [s[1] for s in new_samples]
    new_dataset.classes = desired_class_order
    new_dataset.class_to_idx = {cls: i for i, cls in enumerate(desired_class_order)}
    return new_dataset

def mask_image_blocks(images, mask_ratio=0.1, mask_block_size=(15, 15)):
    """
    Randomly masks a percentage of the image using block masking.
    images: Tensor [B, C, H, W]
    mask_ratio: Fraction of the image area to mask
    mask_block_size: (block_height, block_width)
    """
    B, C, H, W = images.shape
    block_h, block_w = mask_block_size
    total_area = H * W
    block_area = block_h * block_w
    num_blocks_to_mask = int((mask_ratio * total_area) / block_area)

    for i in range(B):
        for _ in range(num_blocks_to_mask):
            top = random.randint(0, H - block_h)
            left = random.randint(0, W - block_w)
            images[i, :, top:top + block_h, left:left + block_w] = 0  # Mask with zeros

    return images
    
def evaluate_vit(model, dataset, mask_ratio=0.1, mask_block_size=(5, 5)):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    dataset = remap_labels(dataset)
    data_loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4)
    criterion = nn.CrossEntropyLoss()
    model.eval()

    loss = 0
    total = 0
    correct = 0

    with torch.no_grad():  # ✅ prevents storing gradients
        for images, labels in tqdm(data_loader, desc=f"Evaluating ViT on FER test data"):
            images = mask_image_blocks(images.clone(), mask_ratio, mask_block_size)
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss += criterion(outputs, labels).item()

            _, preds = outputs.max(1)

            total += labels.size(0)
            correct += preds.eq(labels).sum().item()

            torch.cuda.empty_cache()  # ✅ clear unused memory

    accuracy = 100 * correct / total
    return accuracy, loss

    
def evaluate_cnn(model, dataset, mask_ratio=0.1, mask_block_size=(5, 5)):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss, correct, total = 0, 0, 0
    data = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4)
    with torch.no_grad():
        for images, labels in tqdm(data, desc="Evaluating CNN on FER test data", leave=False):
            images = F.interpolate(images, size=(224, 224), mode='bilinear', align_corners=False)
            # Step 2: Apply masking
            images = mask_image_blocks(images.clone(), mask_ratio, mask_block_size)
            # Step 3: Resize back to 48x48
            images = F.interpolate(images, size=(48, 48), mode='bilinear', align_corners=False)
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss += criterion(outputs, labels).item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    accuracy = 100. * correct / total
    return accuracy, loss

In [8]:
import os
import matplotlib.pyplot as plt

def benchmark_and_plot(cnn_model, vit_model, vit_test_dataset, cnn_test_dataset, mask_ratios, mask_block_sizes):
    # Create directory if it does not exist
    os.makedirs("./comparaison_plots", exist_ok=True)

    # Precompute mask_ratio=0 once
    print("Calculating for mask ratio of 0% (no masking) only once...")
    acc_cnn_0, loss_cnn_0 = evaluate_cnn(cnn_model, cnn_test_dataset, mask_ratio=0, mask_block_size=(1, 1))
    acc_vit_0, loss_vit_0 = evaluate_vit(vit_model, vit_test_dataset, mask_ratio=0, mask_block_size=(1, 1))

    for block_size in mask_block_sizes:
        print(f"\n=== Block Size: {block_size} pixels ===")
        cnn_accs = [acc_cnn_0]
        cnn_losses = [loss_cnn_0]
        vit_accs = [acc_vit_0]
        vit_losses = [loss_vit_0]

        for ratio in mask_ratios[1:]:  # Skip 0, we already did it
            print(f"Calculating for mask ratio of {ratio*100:.0f}%")
            acc_cnn, loss_cnn = evaluate_cnn(cnn_model, cnn_test_dataset, mask_ratio=ratio, mask_block_size=(block_size, block_size))
            acc_vit, loss_vit = evaluate_vit(vit_model, vit_test_dataset, mask_ratio=ratio, mask_block_size=(block_size, block_size))

            cnn_accs.append(acc_cnn)
            cnn_losses.append(loss_cnn)
            vit_accs.append(acc_vit)
            vit_losses.append(loss_vit)

        # Plot Accuracy
        plt.figure(figsize=(8, 5))
        plt.plot(mask_ratios, cnn_accs, marker='o', label="CNN Accuracy")
        plt.plot(mask_ratios, vit_accs, marker='s', label="ViT Accuracy")
        plt.title(f"Accuracy vs Mask Ratio (Block Size {block_size})")
        plt.xlabel("Mask Ratio")
        plt.ylabel("Accuracy (%)")
        plt.legend()
        plt.grid(True)
        acc_filename = f"./comparaison_plots/accuracy_size{block_size}.png"
        plt.savefig(acc_filename, dpi=300, bbox_inches="tight")
        plt.close()

        # Plot Loss
        plt.figure(figsize=(8, 5))
        plt.plot(mask_ratios, cnn_losses, marker='o', label="CNN Loss")
        plt.plot(mask_ratios, vit_losses, marker='s', label="ViT Loss")
        plt.title(f"Loss vs Mask Ratio (Block Size {block_size})")
        plt.xlabel("Mask Ratio")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        loss_filename = f"./comparaison_plots/loss_size{block_size}.png"
        plt.savefig(loss_filename, dpi=300, bbox_inches="tight")
        plt.close()


In [9]:
benchmark_and_plot(
    cnn_model,
    vit_model,
    vit_test_dataset, 
    cnn_test_dataset,
    mask_ratios = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    mask_block_sizes = [10, 20, 30, 40, 50]
)

Calculating for mask ratio of 0% (no masking) only once...


Evaluating ViT on FER test data: 100%|████████████████████████████████████████████| 113/113 [00:20<00:00,  5.41it/s]



=== Block Size: 10 pixels ===
Calculating for mask ratio of 10%


Evaluating ViT on FER test data: 100%|████████████████████████████████████████████| 113/113 [00:22<00:00,  5.02it/s]


Calculating for mask ratio of 20%


...

Evaluating ViT on FER test data: 100%|████████████████████████████████████████████| 113/113 [00:21<00:00,  5.23it/s]


Calculating for mask ratio of 70%


Evaluating ViT on FER test data: 100%|████████████████████████████████████████████| 113/113 [00:21<00:00,  5.22it/s]


Calculating for mask ratio of 80%


Evaluating ViT on FER test data: 100%|████████████████████████████████████████████| 113/113 [00:21<00:00,  5.15it/s]
